In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)


In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import segmentation_models_pytorch as smp
from PIL import Image
import os
import matplotlib.pyplot as plt
import numpy as np
import kagglehub


class SUIMDataset(Dataset):
    def __init__(self, root, img_folder, mask_folder, transform=None):
        self.root = root
        self.img_path = os.path.join(root, img_folder)
        self.mask_path = os.path.join(root, mask_folder)
        self.images = sorted(os.listdir(self.img_path))
        self.masks = sorted(os.listdir(self.mask_path))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.img_path, self.images[idx])).convert("RGB")
        mask = Image.open(os.path.join(self.mask_path, self.masks[idx])).convert("L")

        if self.transform:
            img = self.transform(img)
            mask = transforms.Resize((256, 256), interpolation=Image.NEAREST)(mask)
            mask = torch.from_numpy(np.array(mask))
            mask = remap_mask(mask)

        return img, mask


print(f"Contents of {path}:")
for item in os.listdir(path):
    print(f"  - {item}")

data_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = SUIMDataset(path, 'dataset/images', 'dataset/masks', transform=data_transform)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.imshow(img.permute(1, 2, 0)); plt.title("Image")
plt.subplot(1, 3, 2); plt.imshow(mask, cmap='jet'); plt.title("Ground Truth")
plt.subplot(1, 3, 3); plt.imshow(pred, cmap='jet'); plt.title("Prediction")
plt.show()

In [ ]:
# TO DO
model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
)

In [ ]:
# TO DO
def train_fn(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def valid_fn(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()
    return total_loss / len(loader)


In [ ]:
# TO DO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

train_losses = []
for epoch in range(5):
    loss = train_fn(model, train_loader, optimizer, criterion, device)
    train_losses.append(loss)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}")

plt.plot(train_losses)
plt.title("Training Loss")
plt.show()

In [ ]:
# TO DO

model.eval()
with torch.no_grad():
    img, mask = dataset[0]
    input_img = img.unsqueeze(0).to(device)
    output = model(input_img)
    pred = torch.argmax(output, dim=1).squeeze(0).cpu()

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.imshow(img.permute(1, 2, 0)); plt.title("Image")
plt.subplot(1, 3, 2); plt.imshow(mask, cmap='jet'); plt.title("Ground Truth")
plt.subplot(1, 3, 3); plt.imshow(pred, cmap='jet'); plt.title("Prediction")
plt.show()